## Project Background

Predictive maintenance has become a critical part of smart manufacturing, with industrial machines generating continuous streams of sensor data every day. However, not all machines operate under ideal conditions, some develop faults, some degrade unexpectedly, and some fail entirely with little warning. This problem is not only costly in terms of unplanned downtime and repair costs, but also disrupts production schedules and puts pressure on maintenance teams.

This dataset **Smart Manufacturing IoT-Cloud Monitoring Dataset** is the publicly available, sourced from Kaggle. Ultimately, this project focuses on building a predictive model that classifies whether a machine requires maintenance, using real-time sensor readings such as Temperature, Vibration, Humidity, Pressure, and Energy Consumption to detect early signs of mechanical stress or failure before they escalate into full breakdowns.

---

## Business Problem

Machine failures in industrial settings are unpredictable and negatively impact multiple stakeholders:

### For Organizations
- **Operational costs increase:** Unplanned downtime, emergency repairs, and lost production output drive up costs significantly compared to scheduled maintenance.
- **Production disruptions:** A single machine failure on a production line can cascade, delaying downstream processes and affecting overall output targets.

### For Maintenance & Operations Teams
- **Increased reactive workload:** Without early warning, teams are forced into constant work rather than planned, efficient maintenance scheduling.
- **Resource strain:** Technicians, spare parts inventory, and diagnostic equipment are all pushed to capacity when multiple machines fail around the same time.

### For End-Users / Downstream Stakeholders
- **Delivery delays:** Production slowdowns or halts can result in missed deadlines for goods or components relying on the affected machinery.
- **Safety and trust concerns:** Undetected mechanical failures (e.g. overheating, excessive vibration) can pose safety risks to workers and erode confidence in the reliability of the manufacturing process.

---

## Business Objective
The primary objective of this project is to develop a few machine learning models that predicts whether a machine requires maintenance based on real-time sensor readings.<br>The objectives are:<br>
1. To identify and analyse the key factors that indicate a machine is at risk, such as Temperature, Vibration, and Energy Consumption.
2. To build and compare multiple classification models in order to determine which model performs best.
3. To evaluate model performance using appropriate metrics such as Accuracy, Precision, Recall, F1-score.
4. To generate meaningful insights based on the entire experience of training the model (improvements for future iterations).

# 1. Import Libraries & Data

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/raw/smart_manufacturing_data.csv")

# 2.0 Statistical Exploratory Data Analysis (EDA)
In this section, I will dive into the dataset and perform standard statistical analysis to explore and find out the underlying structure, quality, and characteristics of the data before any cleaning or feature engineering takes place. This includes examining the shape and data types of each column, checking for missing values and duplicates, reviewing summary statistics (mean, median, standard deviation, min/max) for numerical sensor readings, and understanding the distribution of categorical variables such as Machine_Status and Failure_Type. The goal of this stage is purely descriptive, understanding what the data actually looks like and laying the groundwork for the cleaning steps that follows in the next notebook.

In [6]:
df.head()

,timestamp,machine_id,temperature,vibration,humidity,pressure,energy_consumption,machine_status,anomaly_flag,predicted_remaining_life,failure_type,downtime_risk,maintenance_required
0,2025-01-01 00:00:00,39,78.61,28.65,79.96,3.73,2.16,1,0,106,Normal,0.0,0
1,2025-01-01 00:01:00,29,68.19,57.28,35.94,3.64,0.69,1,0,320,Normal,0.0,0
2,2025-01-01 00:02:00,15,98.94,50.20,72.06,1.00,2.49,1,1,19,Normal,1.0,1
3,2025-01-01 00:03:00,43,90.91,37.65,30.34,3.15,4.96,1,1,10,Normal,1.0,1
4,2025-01-01 00:04:00,8,72.32,40.69,56.71,2.68,0.63,2,0,65,Vibration Issue,0.0,1


In [4]:
df.shape

(100000, 13)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   timestamp                 100000 non-null  object 
 1   machine_id                100000 non-null  int64  
 2   temperature               100000 non-null  float64
 3   vibration                 100000 non-null  float64
 4   humidity                  100000 non-null  float64
 5   pressure                  100000 non-null  float64
 6   energy_consumption        100000 non-null  float64
 7   machine_status            100000 non-null  int64  
 8   anomaly_flag              100000 non-null  int64  
 9   predicted_remaining_life  100000 non-null  int64  
 10  failure_type              100000 non-null  object 
 11  downtime_risk             100000 non-null  float64
 12  maintenance_required      100000 non-null  int64  
dtypes: float64(6), int64(5), object(2)
memory usa

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
machine_id,100000.0,25.499330,14.389439,1.00,13.0000,25.00,38.00,50.00
temperature,100000.0,75.015625,10.031884,35.55,68.2675,75.06,81.75,121.94
vibration,100000.0,50.012270,14.985444,-17.09,39.9700,49.96,60.10,113.80
humidity,100000.0,54.995401,14.437960,30.00,42.5200,54.98,67.50,80.00
pressure,100000.0,3.000405,1.152399,1.00,2.0000,3.01,4.00,5.00
energy_consumption,100000.0,2.747064,1.297865,0.50,1.6300,2.74,3.87,5.00
machine_status,100000.0,1.002050,0.446193,0.00,1.0000,1.00,1.00,2.00
anomaly_flag,100000.0,0.089160,0.284976,0.00,0.0000,0.00,0.00,1.00
predicted_remaining_life,100000.0,234.269160,150.063062,1.00,97.0000,230.00,365.00,499.00
downtime_risk,100000.0,0.089155,0.284961,0.00,0.0000,0.00,0.00,1.00


With **100,000 rows** and **13 columns**, this dataset provides a solid foundation for building a reliable classification model. Here are some supporting arguments:

- **Sample size**: 100,000 records is comfortably large for tabular classification tasks well beyond the typical minimum needed for tree-based ensemble models (Random Forest, XGBoost, LightGBM) to learn stable patterns, and large enough to support a proper train/test split (and cross-validation folds within training) without either subset becoming too small to be representative.

- **Class distribution**: While the target variable (`maintenance_required`) is imbalanced, the minority class still contains like 20k records which is enough for the model to learn meaningful patterns from, especially when paired with appropriate handling (e.g. class weighting, or evaluating with Precision/Recall/F1 rather than relying on Accuracy alone).

- **Feature richness**: With 5 continuous sensor readings (Temperature, Vibration, Humidity, Pressure, Energy Consumption), there is enough variety in the feature space for me to do feature engineering for eg: rolling statistics, etc, beyond just the raw sensor values alone.

In [9]:
print(df.isnull().sum())
print("\n")
print(df.duplicated().sum())

timestamp                   0
machine_id                  0
temperature                 0
vibration                   0
humidity                    0
pressure                    0
energy_consumption          0
machine_status              0
anomaly_flag                0
predicted_remaining_life    0
failure_type                0
downtime_risk               0
maintenance_required        0
dtype: int64


0


In [14]:
df.nunique().sort_values()

maintenance_required             2
anomaly_flag                     2
machine_status                   3
failure_type                     5
downtime_risk                    9
machine_id                      50
pressure                       401
energy_consumption             451
predicted_remaining_life       499
humidity                      5001
temperature                   5827
vibration                     8221
timestamp                   100000
dtype: int64

In [11]:
df['maintenance_required'].value_counts(normalize=True)

maintenance_required
0    0.80303
1    0.19697
Name: proportion, dtype: float64

In [16]:
numeric_cols = ['temperature', 'vibration', 'humidity', 'pressure', 'energy_consumption']
df[numeric_cols].std() / df[numeric_cols].mean()

temperature           0.133731
vibration             0.299635
humidity              0.262530
pressure              0.384081
energy_consumption    0.472455
dtype: float64

In [18]:
df[numeric_cols + ['maintenance_required']].corr()

,temperature,vibration,humidity,pressure,energy_consumption,maintenance_required
temperature,1.000000,-0.003251,0.002555,-0.001116,0.003259,0.283021
vibration,-0.003251,1.000000,0.006051,-0.000278,0.007897,0.113072
humidity,0.002555,0.006051,1.000000,-0.000208,-0.002542,-0.003815
pressure,-0.001116,-0.000278,-0.000208,1.000000,-0.008071,-0.001945
energy_consumption,0.003259,0.007897,-0.002542,-0.008071,1.000000,0.001384
maintenance_required,0.283021,0.113072,-0.003815,-0.001945,0.001384,1.000000


In [ ]:
# Compare average sensor readings: healthy machines vs machines needing maintenance
df.groupby('maintenance_required')[numeric_cols].mean()

# Check if each sensor's difference between the two groups is "real" or just random chance
# (p-value < 0.05 = the difference is unlikely to be a coincidence)
from scipy import stats
for col in numeric_cols:
    healthy = df[df['maintenance_required']==0][col]
    needs_maint = df[df['maintenance_required']==1][col]
    t_stat, p_val = stats.ttest_ind(healthy, needs_maint)
    print(f"{col}: p-value = {p_val:.5f}")

temperature: p-value = 0.00000
vibration: p-value = 0.00000
humidity: p-value = 0.22770
pressure: p-value = 0.53848
energy_consumption: p-value = 0.66156


On top of my earlier analysis on data quality, I also checked the Pearson correlation and t-test values to confirm the dataset actually contains meaningful predictive signal before moving forward. Here's why the dataset is fine for model training:

- **Statistical significance**: Hypothesis testing (Section 2) confirmed that Temperature and Vibration show statistically significant differences between healthy and failing machines (p < 0.05), meaning these patterns are unlikely to have occurred by random chance.

- **Domain alignment**: These findings align with real-world industrial knowledge, overheating and excessive vibration are well-established, physically grounded indicators of mechanical failure, which adds confidence that the dataset reflects genuine, realistic patterns rather than arbitrary noise.

- **Multiple validation methods agree**: Since both Pearson correlation and independent t-tests were checked (rather than relying on a single metric), and the weaker linear correlation values were explained by the non-linear, threshold-based nature of failure patterns rather than an absence of signal.

## 2.1 Dataset Efficiency
In this section, I check how the dataset's columns are currently stored by reviewing each column's data type and memory usage. This involves inspecting `df.dtypes` to see how each column is represented, and using `df.memory_usage(deep=True)` to measure how much memory each column actually uses. The goal is to spot columns that may be stored inefficiently and could benefit from conversion to a more suitable data type.

In [20]:
df.dtypes

timestamp                    object
machine_id                    int64
temperature                 float64
vibration                   float64
humidity                    float64
pressure                    float64
energy_consumption          float64
machine_status                int64
anomaly_flag                  int64
predicted_remaining_life      int64
failure_type                 object
downtime_risk               float64
maintenance_required          int64
dtype: object

In [21]:
df.memory_usage(deep=True)

Index                           128
timestamp                   7600000
machine_id                   800000
temperature                  800000
vibration                    800000
humidity                     800000
pressure                     800000
energy_consumption           800000
machine_status               800000
anomaly_flag                 800000
predicted_remaining_life     800000
failure_type                6362029
downtime_risk                800000
maintenance_required         800000
dtype: int64

In [24]:
df['failure_type'].value_counts()

failure_type
Normal              91899
Vibration Issue      3129
Overheating          1989
Pressure Drop        1969
Electrical Fault     1014
Name: count, dtype: int64

Looking at the data types and memory allocation, I identified a few columns that can be optimized:

**1. `timestamp` stored as an object (7.6M bytes)**

This is significantly more than the ~800KB (8 bytes per row) used by other columns of the same length. Since `timestamp` is currently stored as text rather than a proper datetime, it takes up unnecessary memory and prevents any date/time operations (e.g. extracting hour, day of week, etc.). This will be addressed in the cleaning step using pandas' `pd.to_datetime()` function.

**2. `failure_type` stored as an object (6.3M bytes)**

Since this column only contains a handful of repeated text values (e.g. `Normal`, `Vibration Issue`, `Overheating`), converting it to a `category` data type will reduce memory usage and make the dataset more efficient overall, without changing any of the underlying values.

# 3. Storytelling Exploratory Data Analysis
In this section, I will use visualizations to uncover patterns, relationships, and anomalies within the data that aren't immediately obvious from summary statistics alone. Through graphs such as boxplots, correlation heatmaps, and time-based trend lines, I aim to understand how each sensor reading behaves in relation to the target variable (Maintenance_Required), identify potential problems and data quality issues that may need to be addressed during cleaning, and assess whether certain columns (such as Downtime_Risk_Score, Failure_Type, and Predicted_Remaining_Life) show signs of data leakage that would make them unsuitable as input features. The insights drawn from this section will directly inform the decisions made in the data cleaning and feature engineering stages that follow.